# LeetCode 233 — Number of Digit One (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def countDigitOne(self, n: int) -> int:

```

**Mục tiêu học:**

1. Hiểu chính xác đề: đếm **số lần chữ số 1 xuất hiện**, không phải đếm "có bao nhiêu số chứa chữ số 1".
2. Thành thạo bộ công cụ xử lý chữ số: `str(x).count("1")`, `x % 10`, `x // 10`, và cách **tách một số thành 3 khúc `high | cur | low`** tại từng hàng (đơn vị, chục, trăm, ...).
3. Viết được 2 cách **brute force** (vét cạn — thử hết mọi khả năng) để làm "trọng tài" kiểm tra, và hiểu vì sao chúng quá chậm khi `n = 10^9`.
4. Nắm cách chính: **đếm theo từng hàng chữ số** với 3 trường hợp `cur == 0`, `cur == 1`, `cur > 1` — chỉ cần khoảng 10 vòng lặp cho `n = 10^9`, tức `O(log n)`.
5. Hiểu công thức gọn một dòng `(n // (i*10)) * i + min(max(n % (i*10) - i + 1, 0), i)` **từ đâu mà ra**, không học thuộc.
6. Học **digit DP** (quy hoạch động theo chữ số) từ căn bản: đệ quy → memo → cờ `tight`. Đây là "khuôn" tổng quát dùng lại cho rất nhiều bài Hard dạng "đếm các số từ 0 đến n thỏa điều kiện về chữ số".
7. Tránh các lỗi kinh điển: quên `+ 1` khi `cur == 1`, dùng `/` thay `//`, viết `while i < n`, dùng chung memo giữa các test.

**Lộ trình 5 cách** (từ cách người mới tự nghĩ ra đến cách tối ưu / tổng quát):

| Cách | Ý tưởng | Thời gian |
|---|---|---|
| 1 | Brute force: đếm `'1'` trong `str(x)` với mọi `x` từ 0 đến `n` | `O(n log n)` |
| 2 | Brute force: bóc từng chữ số bằng `% 10` và `// 10` | `O(n log n)` |
| 3 | **Đếm theo từng hàng (high / cur / low, 3 trường hợp)** — cách chính | `O(log n)` |
| 4 | Gộp 3 trường hợp thành một công thức gọn | `O(log n)` |
| 5 | Digit DP: đệ quy + memo + cờ `tight` | `O(10 · log² n)` |

## 1. Đề bài

> Given an integer `n`, count the total number of digit `1` appearing in all non-negative integers less than or equal to `n`.

Dịch: Cho một số nguyên `n`. Hãy đếm **tổng số lần chữ số `1` xuất hiện** trong tất cả các số nguyên **không âm** nhỏ hơn hoặc bằng `n`, tức là trong các số `0, 1, 2, ..., n`.

**Ví dụ:**

| Input | Output | Giải thích ngắn |
|---|---:|---|
| `n = 13` | `6` | chữ số 1 xuất hiện trong `1, 10, 11, 12, 13` (số `11` có hai chữ số 1) |
| `n = 0` | `0` | chỉ có số `0`, không có chữ số 1 nào |

**Constraints:**

- `0 <= n <= 10^9`

Ba điều cần để ý ngay từ đầu:

- Đoạn cần xét là `[0, n]` — **tính cả chính `n`**.
- `n` có thể lớn tới `10^9` (một tỉ). Con số này sẽ quyết định cách nào "qua" được LeetCode và cách nào bị **TLE** (Time Limit Exceeded — chạy quá thời gian cho phép).
- Kết quả lớn nhất (khi `n = 10^9`) là `900 000 001`, vẫn vừa kiểu `int` 32-bit của C++/Java. Với Python thì số nguyên không bao giờ tràn, nên không phải lo.

### Hiểu Ví dụ 1 bằng tay (`n = 13`)

Liệt kê các số từ `0` đến `13`, chỉ giữ lại những số có chữ số 1:

| Số | Chữ số 1 nằm ở đâu | Số lần |
|---:|---|---:|
| `1` | **1** | 1 |
| `10` | **1**0 | 1 |
| `11` | **1** **1** | **2** |
| `12` | **1**2 | 1 |
| `13` | **1**3 | 1 |
| | **Tổng** | **6** |

Các số `0, 2, 3, ..., 9` không có chữ số 1 nên đóng góp 0.

**Cẩn thận — cái bẫy đầu tiên:** số `11` có **hai** chữ số 1 nên được tính **2 lần**. Nếu chỉ đếm "có bao nhiêu số chứa chữ số 1" ta được `5` — **sai** với đề. Đề hỏi *"total number of digit 1"* = tổng số **lần xuất hiện**.

Cell dưới kiểm chứng bằng code. Nó dùng `str(x).count("1")` (đếm ký tự `'1'` trong chuỗi của `x`) — mục 2 sẽ giải thích kỹ cú pháp này.

In [1]:
n = 13
total = 0               # tổng số LẦN chữ số 1 xuất hiện (thứ đề hỏi)
numbers_with_one = 0    # số LƯỢNG số có chứa chữ số 1 (KHÔNG phải thứ đề hỏi)

for x in range(n + 1):  # x = 0, 1, ..., 13 (có cả n)
    k = str(x).count("1")
    if k > 0:
        print(f"{x:>2}: có {k} chữ số 1")
        numbers_with_one += 1
    total += k

print("Tổng số lần chữ số 1 xuất hiện:", total)
print("Số lượng số có chứa chữ số 1  :", numbers_with_one)

 1: có 1 chữ số 1
10: có 1 chữ số 1
11: có 2 chữ số 1
12: có 1 chữ số 1
13: có 1 chữ số 1
Tổng số lần chữ số 1 xuất hiện: 6
Số lượng số có chứa chữ số 1  : 5


## 2. Kiến thức nền 1 — Chuỗi, `.count()` và `sum(...)` với generator

**`str(x)`** đổi số thành chuỗi: `str(1101)` cho ra `"1101"` (4 ký tự `'1'`, `'1'`, `'0'`, `'1'`).

**`s.count(t)`** đếm số lần chuỗi con `t` xuất hiện trong `s`:

- `"1101".count("1")` → `3`
- `"2024".count("1")` → `0`

**`range(n + 1)`** cho dãy `0, 1, ..., n`. Nhớ: `range` dừng **trước** giá trị cuối, nên `range(5)` chỉ tới `4`. Muốn có cả `n` phải viết `range(n + 1)`.

**`sum(f(x) for x in ...)`** — *generator expression* (biểu thức sinh): đọc là "tổng của `f(x)` với `x` chạy trong ...". Nó làm đúng việc của vòng `for` + biến `total` nhưng gói trong một dòng:

```python
total = 0
for x in range(14):
    total += str(x).count("1")
# tương đương:
total = sum(str(x).count("1") for x in range(14))
```

In [2]:
print(str(1101), type(str(1101)))   # "1101" là str
print("1101".count("1"))            # 3
print("2024".count("1"))            # 0
print(list(range(5)))               # [0, 1, 2, 3, 4]  : KHÔNG có 5
print(list(range(5 + 1)))           # [0, 1, 2, 3, 4, 5]

total = 0
for x in range(14):
    total += str(x).count("1")
print("Vòng for   :", total)
print("sum(...)   :", sum(str(x).count("1") for x in range(14)))

1101 <class 'str'>
3
0
[0, 1, 2, 3, 4]
[0, 1, 2, 3, 4, 5]
Vòng for   : 6
sum(...)   : 6


## 3. Kiến thức nền 2 — Bóc chữ số bằng `% 10` và `// 10`

Hai phép toán "thần thánh" của mọi bài xử lý chữ số:

- `x % 10` — **phần dư** khi chia cho 10 = **chữ số cuối** (hàng đơn vị). `1101 % 10 = 1`.
- `x // 10` — **chia lấy phần nguyên** = **bỏ chữ số cuối**. `1101 // 10 = 110`.
- Còn `x / 10` là chia **thực**, ra số thập phân `110.1` (kiểu `float`) — **không** dùng để xử lý chữ số.

Lặp lại hai phép trên cho tới khi `x == 0`, ta bóc được lần lượt các chữ số từ **phải sang trái**:

```text
x = 1101 -> chữ số cuối 1, còn lại 110
x = 110  -> chữ số cuối 0, còn lại 11
x = 11   -> chữ số cuối 1, còn lại 1
x = 1    -> chữ số cuối 1, còn lại 0 -> dừng
```

Các chữ số bóc được: `1, 0, 1, 1` → có 3 chữ số 1.

Tổng quát hơn (sẽ dùng ở mục 4): muốn lấy chữ số ở hàng `10^k` thì "bỏ `k` chữ số cuối" rồi lấy chữ số cuối: `(x // 10**k) % 10`.

In [3]:
x = 1101
print(x % 10, x // 10, x / 10)   # 1 110 110.1

while x > 0:
    print(f"x = {x:>4} | chữ số cuối x % 10 = {x % 10} | bỏ chữ số cuối x // 10 = {x // 10}")
    x //= 10

print("Chữ số hàng trăm của 3141592:", (3141592 // 100) % 10)   # 5

1 110 110.1
x = 1101 | chữ số cuối x % 10 = 1 | bỏ chữ số cuối x // 10 = 110
x =  110 | chữ số cuối x % 10 = 0 | bỏ chữ số cuối x // 10 = 11
x =   11 | chữ số cuối x % 10 = 1 | bỏ chữ số cuối x // 10 = 1
x =    1 | chữ số cuối x % 10 = 1 | bỏ chữ số cuối x // 10 = 0
Chữ số hàng trăm của 3141592: 5


## 4. Kiến thức nền 3 — "Hàng" chữ số và cách cắt số thành 3 khúc `high | cur | low`

Ta gọi các vị trí chữ số theo cách học ở tiểu học: hàng **đơn vị**, hàng **chục**, hàng **trăm**, hàng **nghìn**, ... Mỗi hàng có một **giá trị hàng** (place value) `i`:

| Hàng | đơn vị | chục | trăm | nghìn | ... |
|---|---:|---:|---:|---:|---|
| `i` | 1 | 10 | 100 | 1000 | luôn là lũy thừa của 10 |

Chữ số tại hàng `i` của `n` là `(n // i) % 10` (bỏ các chữ số bên phải, rồi lấy chữ số cuối).

**Cắt `n` thành 3 khúc tại hàng `i`** — đây là "nguyên liệu" của cách chính:

```text
n = 3141592, xét hàng trăm (i = 100)

      3141    |    5    |    92
      high    |   cur   |   low
  (bên trái)    (tại i)   (bên phải)
```

| Khúc | Công thức | Ý nghĩa | Với `n = 3141592, i = 100` |
|---|---|---|---|
| `high` | `n // (i * 10)` | các chữ số **bên trái** hàng `i` | `3141592 // 1000 = 3141` |
| `cur` | `(n // i) % 10` | chữ số **tại** hàng `i` | `31415 % 10 = 5` |
| `low` | `n % i` | các chữ số **bên phải** hàng `i` | `3141592 % 100 = 92` |

Ghép lại phải ra đúng `n`: `high * (i * 10) + cur * i + low = 3141 * 1000 + 5 * 100 + 92 = 3141592`.

Hai trường hợp biên dễ quên: ở hàng **đơn vị** thì `low = n % 1 = 0` (không có gì bên phải); ở hàng **cao nhất** thì `high = 0` (không có gì bên trái).

In [4]:
def split_at(n, i):
    """Cắt n tại hàng i (i = 1, 10, 100, ...) thành (high, cur, low)."""
    high = n // (i * 10)
    cur = (n // i) % 10
    low = n % i
    return high, cur, low

print(split_at(3141592, 100))    # (3141, 5, 92)

print("\nCắt n = 214 tại từng hàng:")
for i in [1, 10, 100]:
    high, cur, low = split_at(214, i)
    assert high * (i * 10) + cur * i + low == 214     # ghép lại phải đúng n
    print(f"  i = {i:>3}: high = {high:>2}, cur = {cur}, low = {low:>2}")

(3141, 5, 92)

Cắt n = 214 tại từng hàng:
  i =   1: high = 21, cur = 4, low =  0
  i =  10: high =  2, cur = 1, low =  4
  i = 100: high =  0, cur = 2, low = 14


## 5. Cách 1 — Brute force bằng chuỗi (dùng làm "trọng tài")

Đây là cách mà ai đọc đề xong cũng nghĩ ra ngay: **duyệt mọi số `x` từ `0` đến `n`, đếm số ký tự `'1'` trong `str(x)`, cộng tất cả lại.**

Cách này làm **đúng nguyên văn định nghĩa** của đề, nên gần như không thể sai. Vì vậy ta dùng nó làm **trọng tài** (oracle — "đáp án chuẩn"): mọi cách nhanh hơn ở phía sau đều phải cho **cùng kết quả** với nó trên các `n` nhỏ.

In [5]:
def count_ones_bruteforce_str(n: int) -> int:
    return sum(str(x).count("1") for x in range(n + 1))

for n in [13, 0, 1, 10, 11, 99, 100, 214]:
    print(f"n = {n:>3} -> {count_ones_bruteforce_str(n)}")

n =  13 -> 6
n =   0 -> 0
n =   1 -> 1
n =  10 -> 2
n =  11 -> 4
n =  99 -> 20
n = 100 -> 21
n = 214 -> 147


### Cách 1 vướng ở đâu? Đo thời gian bằng `time.perf_counter`

**Độ phức tạp.** Gọi `d` là số chữ số của `n` (`d ≈ log10(n) + 1`, với `n = 10^9` thì `d = 10`).

- Có `n + 1` số, mỗi số tạo một chuỗi dài tối đa `d` rồi đếm → **Time `O(n · d) = O(n log n)`**.
- Mỗi lúc chỉ giữ một chuỗi tạm → **Space `O(d) = O(log n)`**.

Với `n = 10^9` là khoảng **một tỉ** số, mỗi số lại vài chữ số. Python thường chỉ làm được cỡ vài chục triệu phép đơn giản mỗi giây → riêng việc đi qua một tỉ số đã mất **cỡ một phút trở lên**, trong khi LeetCode chỉ cho phép vài giây → TLE chắc chắn.

Đừng tin suông — hãy **đo**. `time.perf_counter()` trả về "giờ đồng hồ" chính xác (đơn vị giây); lấy giá trị sau trừ giá trị trước là thời gian đã chạy. Vì thời gian tăng gần như **tuyến tính** theo `n`, ta đo với `n = 10^6` rồi nhân 1000 để ước lượng cho `10^9`. (Con số cụ thể tùy máy, nhưng bậc độ lớn thì như nhau.)

In [6]:
import time

for n in [10**5, 10**6]:
    start = time.perf_counter()
    result = count_ones_bruteforce_str(n)
    elapsed = time.perf_counter() - start
    print(f"n = {n:>9,}: kết quả = {result:>9,}, mất {elapsed:.3f} giây")

print(f"Ước lượng cho n = 10^9: khoảng {elapsed * 1000:.0f} giây (≈ {elapsed * 1000 / 60:.1f} phút) -> TLE")

n =   100,000: kết quả =    50,001, mất 0.008 giây
n = 1,000,000: kết quả =   600,001, mất 0.081 giây
Ước lượng cho n = 10^9: khoảng 81 giây (≈ 1.4 phút) -> TLE


## 6. Cách 2 — Brute force bằng `% 10` và `// 10`

Vẫn là "duyệt từng số", nhưng lần này **không dùng chuỗi**: bóc chữ số bằng toán học như ở mục 3. Viết một hàm phụ `ones_in(x)` đếm số chữ số 1 của **một** số `x`, rồi cộng cho mọi `x` từ 0 đến `n`.

Tại sao vẫn nên học cách này dù chưa nhanh hơn?

1. Phỏng vấn hay yêu cầu "không được đổi sang chuỗi".
2. Nó tập cho ta thói quen **nhìn số theo từng chữ số / từng hàng** — chính là bước đệm cho Cách 3.

Lưu ý nhỏ: với `x = 0`, vòng `while x > 0` không chạy lần nào → trả về 0. Đúng, vì số 0 không có chữ số 1.

In [7]:
def ones_in(x: int) -> int:
    """Đếm số chữ số 1 trong x bằng toán học (không dùng str)."""
    count = 0
    while x > 0:
        if x % 10 == 1:
            count += 1
        x //= 10
    return count

def trace_ones_in(x):
    print(f"Đếm chữ số 1 trong {x}:")
    count = 0
    while x > 0:
        d = x % 10
        if d == 1:
            count += 1
            note = f"là 1 -> count = {count}"
        else:
            note = "không phải 1"
        print(f"  x = {x:>5}, chữ số cuối = {d}: {note}")
        x //= 10
    print("  Kết quả:", count)

trace_ones_in(10111)
print(ones_in(0), ones_in(7), ones_in(11), ones_in(2101))   # 0 0 2 2

Đếm chữ số 1 trong 10111:
  x = 10111, chữ số cuối = 1: là 1 -> count = 1
  x =  1011, chữ số cuối = 1: là 1 -> count = 2
  x =   101, chữ số cuối = 1: là 1 -> count = 3
  x =    10, chữ số cuối = 0: không phải 1
  x =     1, chữ số cuối = 1: là 1 -> count = 4
  Kết quả: 4
0 0 2 2


In [8]:
def count_ones_bruteforce_mod(n: int) -> int:
    total = 0
    for x in range(n + 1):
        total += ones_in(x)
    return total

print(count_ones_bruteforce_mod(13), count_ones_bruteforce_mod(0))   # 6 0

# Đối chiếu với trọng tài (Cách 1) cho mọi n từ 0 đến 300
for n in range(301):
    assert count_ones_bruteforce_mod(n) == count_ones_bruteforce_str(n), n
print("Cách 2 khớp Cách 1 với mọi n trong [0, 300]")

6 0
Cách 2 khớp Cách 1 với mọi n trong [0, 300]


## 7. Bước đột phá — Đổi góc nhìn: đếm theo CỘT thay vì theo DÒNG

**Cách 2** có Time `O(n log n)` (y như Cách 1) và Space `O(1)`. Trong Python nó thậm chí còn **chậm hơn** Cách 1 vài lần, vì `str()` và `.count()` chạy bằng mã C bên trong, còn vòng `while` thì Python phải chạy từng lệnh một. Cả hai cách đều vướng **cùng một chỗ**: chúng **duyệt từng số một**. Với `n = 10^9`, chỉ riêng việc "đi qua" một tỉ số đã là quá chậm, dù mỗi số xử lý nhanh tới đâu. Muốn nhanh thật sự, ta phải **đếm mà không liệt kê các số**.

**Mẹo đổi góc nhìn.** Viết tất cả các số từ `0` đến `n` thành một bảng: mỗi **dòng** là một số, các số được **căn phải** và thêm số 0 bên trái cho đủ độ dài (`7` → `07`). Khi đó mỗi **cột** chính là một **hàng chữ số** (cột đơn vị, cột chục, ...).

- Cách 1, 2 đếm **theo từng dòng** (từng số).
- Ta sẽ đếm **theo từng cột** (từng hàng chữ số), rồi cộng các cột lại.

Tổng vẫn y như cũ — giống như cộng một bảng tính theo dòng hay theo cột đều ra cùng một tổng. Thêm số 0 bên trái cũng không làm sai, vì `0` không phải `1`.

**Vì sao đếm theo cột lại có lợi?** Vì chỉ có `d ≤ 10` cột, và (như ta sắp thấy) mỗi cột có **nhịp lặp rất đều** → đếm được bằng **công thức**, không cần duyệt.

Ví dụ với `n = 13` (cell dưới, `s.zfill(2)` là "thêm số 0 bên trái cho đủ 2 ký tự"):

In [9]:
n = 13
width = len(str(n))
tens_ones = units_ones = 0
print(" x | cột chục | cột đơn vị")
for x in range(n + 1):
    s = str(x).zfill(width)       # 7 -> "07"
    tens, units = s[0], s[1]
    tens_ones += (tens == "1")    # True cộng như 1, False như 0
    units_ones += (units == "1")
    print(f"{x:>2} |    {tens}     |     {units}")
print(f"Cột chục có {tens_ones} chữ số 1, cột đơn vị có {units_ones}, tổng = {tens_ones + units_ones}")

 x | cột chục | cột đơn vị
 0 |    0     |     0
 1 |    0     |     1
 2 |    0     |     2
 3 |    0     |     3
 4 |    0     |     4
 5 |    0     |     5
 6 |    0     |     6
 7 |    0     |     7
 8 |    0     |     8
 9 |    0     |     9
10 |    1     |     0
11 |    1     |     1
12 |    1     |     2
13 |    1     |     3
Cột chục có 4 chữ số 1, cột đơn vị có 2, tổng = 6


### Nhịp lặp của từng cột — giống đồng hồ công-tơ-mét

Hãy nhớ tới đồng hồ đo số km (công-tơ-mét) trên xe máy:

- **Bánh xe hàng đơn vị** quay `0 → 9` rồi quay về `0`, cứ **10 số** lại lặp lại. Trong mỗi vòng 10 số, chữ số 1 xuất hiện **đúng 1 lần** (ở số tận cùng bằng 1).
- **Bánh xe hàng chục** đứng yên 10 số rồi mới nhích lên 1 nấc; nó chỉ bằng 1 trong **một đoạn 10 số liền nhau** (`10..19`, `110..119`, ...), và cứ **100 số** lại lặp lại.

Cell dưới xếp các số `0..39` thành lưới 10 cột (dòng `k` chứa `10k .. 10k+9`) và đóng ngoặc `[..]` các số thỏa điều kiện. Nhìn hình sẽ thấy ngay:

- "Hàng đơn vị = 1" là một **cột dọc** → mỗi dòng 10 số có đúng 1 số.
- "Hàng chục = 1" là **trọn một dòng ngang** (`10..19`) → mỗi 100 số có đúng 10 số.

Trong code có `lambda x: x % 10 == 1` — đây là **hàm ngắn không tên**: nhận `x`, trả về `True/False`. Ta truyền nó vào `show_grid` như một "điều kiện".

In [10]:
def show_grid(limit, condition, title):
    print(title)
    for row_start in range(0, limit, 10):
        row = []
        for x in range(row_start, row_start + 10):
            row.append(f"[{x:02d}]" if condition(x) else f" {x:02d} ")
        print(" ".join(row))
    print()

show_grid(40, lambda x: x % 10 == 1, "Các số có chữ số HÀNG ĐƠN VỊ = 1:")
show_grid(40, lambda x: (x // 10) % 10 == 1, "Các số có chữ số HÀNG CHỤC = 1:")

Các số có chữ số HÀNG ĐƠN VỊ = 1:
 00  [01]  02   03   04   05   06   07   08   09 
 10  [11]  12   13   14   15   16   17   18   19 
 20  [21]  22   23   24   25   26   27   28   29 
 30  [31]  32   33   34   35   36   37   38   39 

Các số có chữ số HÀNG CHỤC = 1:
 00   01   02   03   04   05   06   07   08   09 
[10] [11] [12] [13] [14] [15] [16] [17] [18] [19]
 20   21   22   23   24   25   26   27   28   29 
 30   31   32   33   34   35   36   37   38   39 



### Tổng quát: hàng `i` lặp lại theo chu kỳ `10 · i`, mỗi chu kỳ có đúng `i` số

| Hàng | `i` | Độ dài chu kỳ `10·i` | Trong mỗi chu kỳ, các số có chữ số 1 ở hàng này | Bao nhiêu số |
|---|---:|---:|---|---:|
| đơn vị | 1 | 10 | `...1` (đúng 1 số) | 1 |
| chục | 10 | 100 | `..10` → `..19` | 10 |
| trăm | 100 | 1000 | `.100` → `.199` | 100 |
| nghìn | 1000 | 10000 | `1000` → `1999` | 1000 |

**Quy luật:** chia trục số thành các **chu kỳ** dài `10·i` số, bắt đầu từ các bội của `10·i`: `[0, 10i−1]`, `[10i, 20i−1]`, ... Trong **mỗi** chu kỳ, các số có chữ số 1 ở hàng `i` đứng **liền nhau thành một khối** gồm **đúng `i` số**: đó là các số có phần dư (khi chia cho `10·i`) nằm trong `[i, 2i − 1]`.

Ví dụ `i = 10`: chu kỳ `[200, 299]` có khối `[210, 219]` (phần dư `10..19`).

Cell dưới tự tìm các khối liền nhau để kiểm chứng quy luật.

In [11]:
def runs_with_one_at(i, limit):
    """Các khối liên tiếp [a, b] trong 0..limit-1 có chữ số hàng i bằng 1."""
    runs = []
    for x in range(limit):
        if (x // i) % 10 == 1:
            if runs and runs[-1][1] == x - 1:
                runs[-1][1] = x          # nối dài khối đang mở
            else:
                runs.append([x, x])      # mở khối mới
    return runs

print("Hàng chục (i = 10), các số 0..399  :", runs_with_one_at(10, 400))
print("Hàng trăm (i = 100), các số 0..2999:", runs_with_one_at(100, 3000))
print("Độ dài mỗi khối hàng trăm:", [b - a + 1 for a, b in runs_with_one_at(100, 3000)])

Hàng chục (i = 10), các số 0..399  : [[10, 19], [110, 119], [210, 219], [310, 319]]
Hàng trăm (i = 100), các số 0..2999: [[100, 199], [1100, 1199], [2100, 2199]]
Độ dài mỗi khối hàng trăm: [100, 100, 100]


## 8. Đếm tay một cột: hàng chục của `n = 214`

Hàng chục có `i = 10`, chu kỳ dài `100`: `[0..99]`, `[100..199]`, `[200..299]`, ... Số `n = 214` nằm trong chu kỳ thứ ba, và chu kỳ này **đang dở dang** (mới tới 214).

**Bước 1 — Các chu kỳ đầy đủ.** Có `214 // 100 = 2` chu kỳ trọn vẹn (`[0..99]` và `[100..199]`). Con số `2` này chính là **`high`**! Mỗi chu kỳ đầy đủ góp đúng `i = 10` số (`10..19` và `110..119`) → `high · i = 2 · 10 = 20`.

**Bước 2 — Chu kỳ dở dang `[200..214]`.** Khối "hàng chục = 1" của chu kỳ này là `[210..219]`. Ta đã đi tới đâu trong khối đó? Điều này phụ thuộc vào **chữ số hàng chục của `n`**, tức **`cur`**. So sánh 3 giá trị `n` cùng chu kỳ:

| `n` | `cur` | Chu kỳ dở dang | Phần khối `[210..219]` nằm trong `[200..n]` | Đóng góp | Tổng hàng chục |
|---:|---:|---|---|---:|---:|
| 204 | 0 | `[200..204]` | chưa chạm tới 210 → không có số nào | `0` | 20 |
| 214 | 1 | `[200..214]` | `210..214` → **5 số** = `low + 1` | `5` | 25 |
| 254 | 5 | `[200..254]` | trọn khối `210..219` → **10 số** = `i` | `10` | 30 |

**Vì sao `cur == 1` lại cho `low + 1`?** Khi `cur == 1`, ta đang đứng **giữa khối**: các số trong khối từ `210` tới `214` có phần bên phải (`low`) chạy từ `0` tới `4`. Đếm từ `0` đến `low` có **`low + 1`** số (đếm cả số 0!) → `4 + 1 = 5`. Đây chính là chỗ người mới hay quên `+ 1`.

In [12]:
def count_one_at_position_brute(n, i):
    """Trọng tài cho MỘT cột: đếm các x trong [0, n] có chữ số hàng i bằng 1."""
    return sum(1 for x in range(n + 1) if (x // i) % 10 == 1)

for n in [204, 214, 254]:
    high, cur, low = split_at(n, 10)
    print(f"n = {n}: high = {high}, cur = {cur}, low = {low} "
          f"-> hàng chục có {count_one_at_position_brute(n, 10)} chữ số 1")

n = 204: high = 2, cur = 0, low = 4 -> hàng chục có 20 chữ số 1
n = 214: high = 2, cur = 1, low = 4 -> hàng chục có 25 chữ số 1
n = 254: high = 2, cur = 5, low = 4 -> hàng chục có 30 chữ số 1


## 9. Công thức 3 trường hợp cho một hàng `i` + bảng đếm tay

Gộp hai bước ở mục 8 lại, số chữ số 1 nằm ở hàng `i` (gọi là `count_i`) là:

```text
nếu cur == 0:   count_i = high * i                  (chu kỳ dở dang chưa chạm khối)
nếu cur == 1:   count_i = high * i + (low + 1)      (đang đứng giữa khối)
nếu cur >= 2:   count_i = (high + 1) * i            (đã đi qua trọn khối)
```

Nói bằng lời: phần **`high * i`** lúc nào cũng có (từ các chu kỳ đầy đủ); chu kỳ dở dang góp thêm **`0`**, **`low + 1`**, hoặc **`i`** tùy `cur`.

Đáp án cuối cùng = **tổng `count_i` qua mọi hàng** `i = 1, 10, 100, ...`

**Bảng đếm tay `n = 13`:**

| Hàng | `i` | `high` | `cur` | `low` | Trường hợp | `count_i` |
|---|---:|---:|---:|---:|---|---:|
| đơn vị | 1 | 1 | 3 | 0 | `cur ≥ 2` | `(1 + 1) · 1 = 2` |
| chục | 10 | 0 | 1 | 3 | `cur = 1` | `0 · 10 + 3 + 1 = 4` |
| | | | | | **Tổng** | **6** (đúng Ví dụ 1) |

Đối chiếu với mục 7: cột đơn vị có 2 (số `01`, `11`), cột chục có 4 (`10..13`).

**Bảng đếm tay `n = 214`:**

| Hàng | `i` | `high` | `cur` | `low` | Trường hợp | `count_i` |
|---|---:|---:|---:|---:|---|---:|
| đơn vị | 1 | 21 | 4 | 0 | `cur ≥ 2` | `(21 + 1) · 1 = 22` |
| chục | 10 | 2 | 1 | 4 | `cur = 1` | `2 · 10 + 4 + 1 = 25` |
| trăm | 100 | 0 | 2 | 14 | `cur ≥ 2` | `(0 + 1) · 100 = 100` |
| | | | | | **Tổng** | **147** |

Ý nghĩa từng dòng: hàng đơn vị có `22` số `1, 11, 21, ..., 211` (dạng `x1` với `x = 0..21`); hàng chục có `25` số như mục 8; hàng trăm có trọn khối `100..199`.

In [13]:
def count_at_position_formula(n, i):
    high, cur, low = split_at(n, i)
    if cur == 0:
        return high * i
    elif cur == 1:
        return high * i + low + 1
    else:
        return (high + 1) * i

for n in [13, 214]:
    print(f"n = {n}")
    total = 0
    i = 1
    while i <= n:
        f = count_at_position_formula(n, i)
        b = count_one_at_position_brute(n, i)
        print(f"   i = {i:>3}: công thức = {f:>3}, brute force = {b:>3}")
        assert f == b
        total += f
        i *= 10
    print(f"   Tổng = {total}, trọng tài Cách 1 = {count_ones_bruteforce_str(n)}")

n = 13
   i =   1: công thức =   2, brute force =   2
   i =  10: công thức =   4, brute force =   4
   Tổng = 6, trọng tài Cách 1 = 6
n = 214
   i =   1: công thức =  22, brute force =  22
   i =  10: công thức =  25, brute force =  25
   i = 100: công thức = 100, brute force = 100
   Tổng = 147, trọng tài Cách 1 = 147


## 10. Cách 3 (cách chính) — Đếm theo từng hàng chữ số

Ghép mọi mảnh lại thành thuật toán:

1. `ans = 0`, `i = 1` (bắt đầu từ hàng đơn vị).
2. Trong khi `i <= n`:
   - Cắt `n` tại hàng `i`: `high = n // (i*10)`, `cur = (n // i) % 10`, `low = n % i`.
   - Cộng `count_i` vào `ans` theo 3 trường hợp của `cur`.
   - `i *= 10` để sang hàng kế tiếp.
3. Trả về `ans`.

**Vì sao điều kiện là `i <= n` (có dấu bằng)?** Hàng `i` chỉ "tồn tại" khi `n` có chữ số ở hàng đó, tức là `n >= i`. Các hàng cao hơn chữ số đầu tiên của `n` thì **mọi** số từ 0 đến `n` đều mang chữ số 0 ở đó → không góp chữ số 1 nào.

- `n = 214`: chạy `i = 1, 10, 100`; tới `i = 1000 > 214` thì dừng.
- `n = 100`: **phải** chạy cả `i = 100` (lúc này `cur = 1`, đếm được chính số 100). Nếu viết `i < n` sẽ bỏ sót — xem Lỗi 2 ở mục 17.
- `n = 0`: vòng lặp không chạy lần nào → trả về `0` = Ví dụ 2. **Không cần** xử lý riêng.

In [14]:
def count_digit_one_position(n: int) -> int:
    ans = 0
    i = 1                          # giá trị hàng: 1, 10, 100, ...
    while i <= n:
        high = n // (i * 10)       # khúc bên trái hàng i
        cur = (n // i) % 10        # chữ số tại hàng i
        low = n % i                # khúc bên phải hàng i
        if cur == 0:
            ans += high * i
        elif cur == 1:
            ans += high * i + low + 1
        else:
            ans += (high + 1) * i
        i *= 10
    return ans

for n in [13, 0, 214, 1024, 10**9]:
    print(f"n = {n:>10} -> {count_digit_one_position(n)}")

n =         13 -> 6
n =          0 -> 0
n =        214 -> 147
n =       1024 -> 338
n = 1000000000 -> 900000001


## 11. Mô phỏng từng bước — `n = 1024` (đủ cả 3 trường hợp)

Chọn `1024` vì các chữ số của nó là `1, 0, 2, 4` → gặp đủ **`cur = 0`** (hàng trăm), **`cur = 1`** (hàng nghìn) và **`cur ≥ 2`** (hàng đơn vị, hàng chục).

| Bước | `i` | `high = n // (10i)` | `cur = (n // i) % 10` | `low = n % i` | Trường hợp | Cộng thêm | `ans` |
|---:|---:|---:|---:|---:|---|---|---:|
| 1 | 1 | 102 | 4 | 0 | `cur ≥ 2` | `(102 + 1) · 1 = 103` | 103 |
| 2 | 10 | 10 | 2 | 4 | `cur ≥ 2` | `(10 + 1) · 10 = 110` | 213 |
| 3 | 100 | 1 | 0 | 24 | `cur = 0` | `1 · 100 = 100` | 313 |
| 4 | 1000 | 0 | 1 | 24 | `cur = 1` | `0 · 1000 + 24 + 1 = 25` | 338 |
| — | 10000 | | | | `10000 > 1024` → dừng | | **338** |

Giải thích hai dòng "đặc biệt":

- **Bước 3 (`cur = 0`)**: các số có hàng trăm = 1 trong `[0, 1024]` chỉ là `100..199` (một chu kỳ đầy đủ `[0..999]`). Chu kỳ dở dang `[1000..1024]` có khối `[1100..1199]` — **chưa chạm tới** → góp 0.
- **Bước 4 (`cur = 1`)**: hàng nghìn = 1 với các số `1000..1024` → `24 + 1 = 25` số (đếm cả `1000`).

Hàm `trace_position` in đúng bảng trên, sau đó chạy thêm `214`, `13` và `0`.

In [15]:
def trace_position(n):
    print(f"n = {n}")
    ans = 0
    i = 1
    step = 1
    while i <= n:
        high, cur, low = n // (i * 10), (n // i) % 10, n % i
        if cur == 0:
            add, case = high * i, "cur = 0 -> high*i"
        elif cur == 1:
            add, case = high * i + low + 1, "cur = 1 -> high*i + low + 1"
        else:
            add, case = (high + 1) * i, "cur > 1 -> (high+1)*i"
        ans += add
        print(f"  Bước {step}: i = {i:>4} | high = {high:>3}, cur = {cur}, low = {low:>2} "
              f"| {case:<27} | +{add:<4} -> ans = {ans}")
        i *= 10
        step += 1
    print(f"  i = {i} > n -> dừng, trả về {ans}")
    return ans

trace_position(1024)
print()
trace_position(214)
print()
trace_position(13)
print()
_ = trace_position(0)       # gán vào _ để notebook không in thêm giá trị trả về

n = 1024
  Bước 1: i =    1 | high = 102, cur = 4, low =  0 | cur > 1 -> (high+1)*i       | +103  -> ans = 103
  Bước 2: i =   10 | high =  10, cur = 2, low =  4 | cur > 1 -> (high+1)*i       | +110  -> ans = 213
  Bước 3: i =  100 | high =   1, cur = 0, low = 24 | cur = 0 -> high*i           | +100  -> ans = 313
  Bước 4: i = 1000 | high =   0, cur = 1, low = 24 | cur = 1 -> high*i + low + 1 | +25   -> ans = 338
  i = 10000 > n -> dừng, trả về 338

n = 214
  Bước 1: i =    1 | high =  21, cur = 4, low =  0 | cur > 1 -> (high+1)*i       | +22   -> ans = 22
  Bước 2: i =   10 | high =   2, cur = 1, low =  4 | cur = 1 -> high*i + low + 1 | +25   -> ans = 47
  Bước 3: i =  100 | high =   0, cur = 2, low = 14 | cur > 1 -> (high+1)*i       | +100  -> ans = 147
  i = 1000 > n -> dừng, trả về 147

n = 13
  Bước 1: i =    1 | high =   1, cur = 3, low =  0 | cur > 1 -> (high+1)*i       | +2    -> ans = 2
  Bước 2: i =   10 | high =   0, cur = 1, low =  3 | cur = 1 -> high*i + low + 1 | +4    ->

### Nhận xét từ trace và độ phức tạp của Cách 3

- Số vòng lặp = **số chữ số `d` của `n`**: `1024` → 4 vòng, `214` → 3 vòng, `13` → 2 vòng, `0` → 0 vòng.
- Mỗi vòng chỉ vài phép `//`, `%`, `*`, `+` → `O(1)`.
- **Time: `O(d) = O(log n)`**. Với `n = 10^9` chỉ có **10 vòng** — so với khoảng một tỉ lần lặp của brute force.
- **Space: `O(1)`** — chỉ dùng vài biến `ans, i, high, cur, low`.

Một chi tiết thú vị khi `n = 10^9`: ở vòng thứ 10, `i = 10^9` có `high = 0, cur = 1, low = 0` → cộng `0 + 0 + 1 = 1`, chính là chữ số 1 đứng đầu của số `1 000 000 000`.

## 12. Cách 4 — Gộp 3 trường hợp thành một công thức gọn

Trên mạng hay gặp lời giải chỉ có một dòng trong vòng lặp:

```python
ans += (n // (i * 10)) * i + min(max(n % (i * 10) - i + 1, 0), i)
```

Nhìn thì "bí hiểm", nhưng thật ra đó chính là Cách 3 viết gọn. Ta suy ra nó từ ý tưởng **chu kỳ** ở mục 7–8:

- **Khúc đầu** `(n // (i * 10)) * i` = `high * i`: phần của các chu kỳ đầy đủ — y như cũ.
- **Khúc sau** lo chu kỳ dở dang. Đặt **`r = n % (i * 10)`** — "đã đi được bao xa trong chu kỳ dở dang" (`r` từ `0` đến `10i − 1`). Ví dụ `n = 214, i = 10`: chu kỳ dở dang là `[200..299]`, ta đang ở vị trí `r = 14` của nó.
  - Chu kỳ dở dang chứa các vị trí `0, 1, ..., r` (tương ứng `200..214`).
  - Khối chữ số 1 chiếm các vị trí `i .. 2i − 1` (tương ứng `210..219`).
  - Số phần tử chung của `[0, r]` và `[i, 2i − 1]`:
    - nếu `r < i`: chưa chạm khối → `0`
    - nếu `i ≤ r ≤ 2i − 1`: chung đoạn `[i, r]` → `r − i + 1`
    - nếu `r ≥ 2i`: chứa trọn khối → `i`
- Ba trường hợp trên viết gọn thành `min(max(r − i + 1, 0), i)`: lấy giá trị "thô" `r − i + 1` rồi **kẹp** nó vào đoạn `[0, i]`.

### Kiến thức nền 4 — "Kẹp" (clamp) một giá trị bằng `min(max(x, lo), hi)`

Đọc từ trong ra ngoài:

- `max(x, lo)`: nếu `x` nhỏ hơn `lo` thì nâng lên thành `lo` (ở đây: số âm thành 0).
- `min(..., hi)`: nếu vượt `hi` thì hạ xuống thành `hi` (ở đây: lớn hơn `i` thành `i`).
- Nếu `x` vốn nằm trong `[lo, hi]` thì giữ nguyên.

In [16]:
def clamp(x, lo, hi):
    return min(max(x, lo), hi)

for x in [-7, 0, 3, 10, 15]:
    print(f"clamp({x:>2}, 0, 10) = {clamp(x, 0, 10)}")

clamp(-7, 0, 10) = 0
clamp( 0, 0, 10) = 0
clamp( 3, 0, 10) = 3
clamp(10, 0, 10) = 10
clamp(15, 0, 10) = 10


### Công thức gọn có thật sự khớp 3 trường hợp? + độ phức tạp

Vì `r = n % (10i) = cur · i + low` với `0 ≤ low ≤ i − 1`, ta xét từng giá trị `cur`:

| `cur` | `r = cur·i + low` | `r − i + 1` | Sau khi kẹp vào `[0, i]` | Khớp Cách 3 |
|---|---|---|---|---|
| `0` | `low` (nhỏ hơn `i`) | `low − i + 1 ≤ 0` | `0` | `cur = 0`: thêm 0 |
| `1` | `i + low` | `low + 1` (từ 1 đến `i`) | `low + 1` | `cur = 1`: thêm `low + 1` |
| `≥ 2` | `≥ 2i` | `≥ i + 1` | `i` | `cur ≥ 2`: thêm `i` |

Vậy Cách 4 **chính là** Cách 3, chỉ khác cách viết. Ví dụ `n = 214, i = 10`: `r = 14`, `r − i + 1 = 5`, kẹp vẫn là `5` = `low + 1`.

**Độ phức tạp:** như Cách 3 — **Time `O(log n)`, Space `O(1)`**.

**Lời khuyên:** khi phỏng vấn, hãy trình bày Cách 3 (3 trường hợp dễ giải thích), sau đó mới nói "có thể gộp thành một dòng". Đừng học thuộc Cách 4 khi chưa hiểu ý "giao của hai đoạn" — rất dễ nhớ nhầm `+1`, `−1`.

In [17]:
def count_digit_one_formula(n: int) -> int:
    ans = 0
    i = 1
    while i <= n:
        ans += (n // (i * 10)) * i + min(max(n % (i * 10) - i + 1, 0), i)
        i *= 10
    return ans

# Xem từng thành phần của công thức với n = 1024
n = 1024
i = 1
while i <= n:
    full = (n // (i * 10)) * i
    r = n % (i * 10)
    raw = r - i + 1
    part = min(max(raw, 0), i)
    print(f"i = {i:>4}: full = {full:>3}, r = {r:>4}, r - i + 1 = {raw:>4}, kẹp -> {part:>3}, cộng {full + part}")
    i *= 10

print(count_digit_one_formula(13), count_digit_one_formula(0), count_digit_one_formula(1024))   # 6 0 338

i =    1: full = 102, r =    4, r - i + 1 =    4, kẹp ->   1, cộng 103
i =   10: full = 100, r =   24, r - i + 1 =   15, kẹp ->  10, cộng 110
i =  100: full = 100, r =   24, r - i + 1 =  -75, kẹp ->   0, cộng 100
i = 1000: full =   0, r = 1024, r - i + 1 =   25, kẹp ->  25, cộng 25
6 0 338


## 13. Cách 5 — Digit DP: kỹ thuật tổng quát cho bài "đếm trên các số từ 0 đến n"

Đã có Cách 3 chạy `O(log n)`, tại sao còn học thêm?

Vì Cách 3 dựa vào một tính chất **may mắn** của riêng bài này: mỗi hàng chữ số đếm được **độc lập** bằng một công thức đẹp. Rất nhiều bài "họ hàng" **không** có công thức như vậy, ví dụ:

- Đếm các số `≤ n` mà biểu diễn nhị phân không có hai bit 1 liền nhau (LeetCode 600).
- Đếm các số `≤ n` có các chữ số đôi một khác nhau (LeetCode 2376).
- Đếm các số trong `[L, R]` có tổng chữ số nằm trong `[a, b]` (LeetCode 2719).

**Digit DP** (quy hoạch động theo chữ số) là "công thức nấu ăn" chung cho cả họ bài này: **xây số từng chữ số một từ trái sang phải**, chỉ nhớ **vừa đủ thông tin** (gọi là *trạng thái*), và dùng **memo** để không tính lại.

Cần hai kiến thức nền, ta học từ đầu ngay sau đây: **(1) đệ quy**, **(2) memoization**.

### Kiến thức nền 5 — Đệ quy (recursion)

**Đệ quy** là khi một hàm **gọi lại chính nó** trên một bài toán **nhỏ hơn**. Mỗi hàm đệ quy có 2 phần:

- **Base case** (trường hợp dừng): bài đủ nhỏ để trả lời ngay, không gọi tiếp.
- **Recursive case** (bước đệ quy): thu nhỏ bài toán, gọi lại hàm, rồi kết hợp kết quả.

Ví dụ: đếm chữ số 1 trong `x` theo kiểu đệ quy:

```text
ones(x) = 0                                              nếu x == 0   (base case)
ones(x) = (1 nếu x % 10 == 1, ngược lại 0) + ones(x // 10)            (bóc chữ số cuối, phần còn lại giao cho đệ quy)
```

Tính tay: `ones(1101) = 1 + ones(110) = 1 + 0 + ones(11) = 1 + 0 + 1 + ones(1) = 1 + 0 + 1 + 1 + ones(0) = 3`.

Khi chạy, mỗi lời gọi phải **chờ** lời gọi bên trong trả về rồi mới tính tiếp — các lời gọi đang chờ xếp chồng lên nhau gọi là **call stack** (ngăn xếp lời gọi). Trong trace dưới đây, độ thụt lề = độ sâu của call stack.

In [18]:
def ones_recursive(x, depth=0):
    indent = "    " * depth
    if x == 0:
        print(f"{indent}ones(0) = 0   <- base case, bắt đầu trả về")
        return 0
    last = 1 if x % 10 == 1 else 0
    print(f"{indent}ones({x}) = {last} + ones({x // 10})")
    result = last + ones_recursive(x // 10, depth + 1)
    print(f"{indent}ones({x}) trả về {result}")
    return result

_ = ones_recursive(1101)

ones(1101) = 1 + ones(110)
    ones(110) = 0 + ones(11)
        ones(11) = 1 + ones(1)
            ones(1) = 1 + ones(0)
                ones(0) = 0   <- base case, bắt đầu trả về
            ones(1) trả về 1
        ones(11) trả về 2
    ones(110) trả về 2
ones(1101) trả về 3


### Kiến thức nền 6 — Memoization với `functools.lru_cache`

Đệ quy có một nhược điểm: **cùng một bài con có thể bị gọi rất nhiều lần**. Ví dụ kinh điển là Fibonacci: `fib(k) = fib(k − 1) + fib(k − 2)`. Để tính `fib(25)`, `fib(23)` bị tính 2 lần, `fib(22)` 3 lần, ... tổng cộng hơn hai trăm nghìn lời gọi!

**Memoization** (ghi nhớ): lần đầu tính xong `f(args)` thì **lưu kết quả** vào một dict với key là `args`; lần sau gặp lại đúng `args` thì **lấy ra luôn**, không tính lại.

Trong Python, không cần tự viết dict: đặt **decorator** `@lru_cache(maxsize=None)` ngay trên dòng `def`. *Decorator* là một "lớp vỏ" gắn thêm tính năng cho hàm — ở đây là bộ nhớ đệm (cache). Lưu ý:

- Các tham số phải **hashable** (dùng được làm key của dict): `int`, `bool`, `str`, `tuple` thì được; `list` thì không.
- `.cache_info()` cho biết số lần trúng cache (`hits`) / phải tính thật (`misses`); `.cache_clear()` xóa cache.

Trong cell dưới, `global calls` cho phép hàm sửa biến đếm `calls` nằm bên ngoài hàm.

In [19]:
from functools import lru_cache

calls = 0
def fib_plain(k):
    global calls
    calls += 1
    if k < 2:
        return k
    return fib_plain(k - 1) + fib_plain(k - 2)

print("fib(25) không memo =", fib_plain(25), "| số lần gọi hàm:", calls)

calls = 0
@lru_cache(maxsize=None)
def fib_memo(k):
    global calls
    calls += 1
    if k < 2:
        return k
    return fib_memo(k - 1) + fib_memo(k - 2)

print("fib(25) có memo    =", fib_memo(25), "| số lần THỰC SỰ tính:", calls)
print(fib_memo.cache_info())

fib(25) không memo = 75025 | số lần gọi hàm: 242785
fib(25) có memo    = 75025 | số lần THỰC SỰ tính: 26
CacheInfo(hits=23, misses=26, maxsize=None, currsize=26)


### Digit DP bước 1 — Bài khởi động: đếm có bao nhiêu số trong `[0, n]`

Đáp án hiển nhiên là `n + 1`. Nhưng chính vì biết trước đáp án, đây là bài hoàn hảo để **làm quen bộ máy** digit DP.

**Hình dung "điền ô".** Với `n = 214`, vẽ 3 ô `[2][1][4]`. Mọi số `x` trong `[0, 214]` đều viết được bằng đúng 3 chữ số nếu thêm số 0 bên trái (`7` → `007`, `56` → `056`). Mỗi số ↔ **đúng một** cách điền 3 ô sao cho chuỗi điền được `≤ "214"`. Vậy đếm số ⇔ đếm cách điền ô.

Ta điền **từ trái sang phải**. Ở mỗi ô, câu hỏi then chốt là: *được phép điền chữ số nào?* Câu trả lời phụ thuộc vào cờ **`tight`**:

- **`tight = True`** ("còn dính trần"): các chữ số đã điền **trùng khít** với phần đầu của `n` (ví dụ đã điền `2`, rồi `1`). Khi đó ô tiếp theo chỉ được điền **từ 0 đến chữ số của `n` tại ô đó** — điền lớn hơn là vượt quá `n`.
- **`tight = False`** ("đã thoát trần"): đã có một ô nào đó điền **nhỏ hơn** chữ số của `n` (ví dụ ô đầu điền `1` → số dạng `1__` chắc chắn `< 2__`). Khi đó các ô sau **tự do** `0..9`.

Cập nhật cờ: `tight_mới = tight and (d == limit)`. Một khi đã thoát trần thì **thoát mãi mãi**.

```text
ô 0 (tight, giới hạn 2):  chọn 0 -> "0__" thoát trần, 2 ô sau tự do: 10 × 10 = 100 số (000..099)
                          chọn 1 -> "1__" thoát trần: 100 số (100..199)
                          chọn 2 -> "2__" VẪN tight  -> sang ô 1 với giới hạn 1
ô 1 (tight, giới hạn 1):  chọn 0 -> "20_" thoát trần: 10 số (200..209)
                          chọn 1 -> "21_" VẪN tight  -> sang ô 2 với giới hạn 4
ô 2 (tight, giới hạn 4):  chọn 0..4 -> 210..214: 5 số
Tổng: 100 + 100 + 10 + 5 = 215 = 214 + 1
```

**Trạng thái:** `dfs(pos, tight)` = số cách điền các ô từ `pos` tới cuối. **Base case:** `pos == L` (đã điền đủ `L` ô) → đúng **1** cách (một số hoàn chỉnh).

Trong code có `[int(c) for c in str(n)]` — *list comprehension*: duyệt từng ký tự `c` của chuỗi và đổi thành số, `214` → `[2, 1, 4]`.

In [20]:
def count_numbers_upto(n: int) -> int:
    digits = [int(c) for c in str(n)]      # 214 -> [2, 1, 4]
    L = len(digits)

    @lru_cache(maxsize=None)
    def dfs(pos, tight):
        if pos == L:                        # đã điền đủ L ô -> đúng 1 số
            return 1
        limit = digits[pos] if tight else 9
        total = 0
        for d in range(limit + 1):
            total += dfs(pos + 1, tight and d == limit)
        return total

    return dfs(0, True)

print([int(c) for c in str(214)])
for n in [0, 7, 13, 214, 1024, 10**9]:
    got = count_numbers_upto(n)
    assert got == n + 1
    print(f"n = {n:>10}: digit DP đếm được {got:>10}, n + 1 = {n + 1:>10}")

[2, 1, 4]
n =          0: digit DP đếm được          1, n + 1 =          1
n =          7: digit DP đếm được          8, n + 1 =          8
n =         13: digit DP đếm được         14, n + 1 =         14
n =        214: digit DP đếm được        215, n + 1 =        215
n =       1024: digit DP đếm được       1025, n + 1 =       1025
n = 1000000000: digit DP đếm được 1000000001, n + 1 = 1000000001


### Digit DP bước 2 — Thêm trạng thái "đã có bao nhiêu chữ số 1"

Bây giờ không đếm **số lượng số** nữa, mà cộng **tổng số chữ số 1** của tất cả các số. Ta thêm vào trạng thái biến `ones` = số chữ số 1 trong các ô **đã điền**.

**Định nghĩa:** `dfs(pos, ones, tight)` = **tổng số chữ số 1** của tất cả các số hoàn chỉnh có thể tạo ra, biết rằng: đã điền xong các ô `0..pos−1`, trong đó có `ones` chữ số 1, và cờ `tight` như bước 1.

- **Base case:** `pos == L` → số đã hoàn chỉnh, nó có đúng `ones` chữ số 1 → trả về `ones`.
- **Chuyển trạng thái:** với mỗi `d` từ `0` tới `limit`, gọi `dfs(pos + 1, ones + (1 nếu d == 1 ngược lại 0), tight and d == limit)` rồi cộng lại.
- **Đáp án:** `dfs(0, 0, True)`.

**Số 0 đứng đầu có làm sai không?** Không. Số `7` được điền là `"07"`; chữ số 0 thêm vào không phải 1 nên `ones` không đổi. (Nếu bài hỏi đếm chữ số **0** thì khác — xem mục Biến thể.)

Hãy trace `n = 13` (2 ô `[1][3]`) **không dùng memo** để thấy toàn bộ cây lời gọi. Mỗi "lá" là một số hoàn chỉnh.

In [21]:
def trace_digit_dp(n):
    digits = [int(c) for c in str(n)]
    L = len(digits)

    def dfs(pos, ones, tight, prefix):
        indent = "    " * pos
        if pos == L:
            print(f"{indent}lá '{prefix}': có {ones} chữ số 1")
            return ones
        limit = digits[pos] if tight else 9
        print(f"{indent}dfs(pos={pos}, ones={ones}, tight={tight}) prefix='{prefix}' -> thử d = 0..{limit}")
        total = 0
        for d in range(limit + 1):
            total += dfs(pos + 1, ones + (1 if d == 1 else 0), tight and d == limit, prefix + str(d))
        print(f"{indent}=> dfs(pos={pos}, ones={ones}, tight={tight}) trả về {total}")
        return total

    return dfs(0, 0, True, "")

print("Kết quả:", trace_digit_dp(13))

dfs(pos=0, ones=0, tight=True) prefix='' -> thử d = 0..1
    dfs(pos=1, ones=0, tight=False) prefix='0' -> thử d = 0..9
        lá '00': có 0 chữ số 1
        lá '01': có 1 chữ số 1
        lá '02': có 0 chữ số 1
        lá '03': có 0 chữ số 1
        lá '04': có 0 chữ số 1
        lá '05': có 0 chữ số 1
        lá '06': có 0 chữ số 1
        lá '07': có 0 chữ số 1
        lá '08': có 0 chữ số 1
        lá '09': có 0 chữ số 1
    => dfs(pos=1, ones=0, tight=False) trả về 1
    dfs(pos=1, ones=1, tight=True) prefix='1' -> thử d = 0..3
        lá '10': có 1 chữ số 1
        lá '11': có 2 chữ số 1
        lá '12': có 1 chữ số 1
        lá '13': có 1 chữ số 1
    => dfs(pos=1, ones=1, tight=True) trả về 5
=> dfs(pos=0, ones=0, tight=True) trả về 6
Kết quả: 6


### Đọc trace, vì sao cần memo, và độ phức tạp của Cách 5

**Đọc trace `n = 13`:**

- Gốc `dfs(pos=0, ones=0, tight=True)`: giới hạn là chữ số đầu của 13 → thử `d = 0` và `d = 1`.
- `d = 0` → prefix `"0"`, **thoát trần** → ô thứ hai tự do `0..9` → các số `00..09` (tức `0..9`), chỉ `"01"` có một chữ số 1 → trả về `1`.
- `d = 1` → prefix `"1"`, `ones = 1`, **vẫn tight** (vì `1 == limit`) → ô thứ hai chỉ được `0..3` → các số `10, 11, 12, 13` có lần lượt `1, 2, 1, 1` chữ số 1 → trả về `5`.
- Gốc cộng lại: `1 + 5 = 6`.

**Vì sao cần memo?** Không có memo, cây có khoảng `n` lá (mỗi lá là một số trong `[0, n]`) → chậm **y như brute force**! Nhưng rất nhiều lời gọi **trùng trạng thái**. Ví dụ với `n = 99999`, trạng thái `dfs(3, 0, False)` được gọi từ các prefix `"000"`, `"020"`, `"202"`, `"222"`, ... Tất cả đều hỏi cùng một câu: *"2 ô cuối tự do, tổng số chữ số 1 (cộng thêm 0 cái đã có) là bao nhiêu?"* — câu trả lời không phụ thuộc prefix cụ thể, chỉ phụ thuộc `(pos, ones, tight)`. Memo tính mỗi trạng thái **một lần**.

**Đếm trạng thái:** `pos` có `d + 1` giá trị, `ones` có tối đa `d + 1` giá trị, `tight` có 2 giá trị → nhiều nhất `2(d + 1)²` trạng thái, mỗi trạng thái lặp tối đa 10 chữ số.

- **Time: `O(10 · d²) = O(log² n)`** — với `d = 10` chỉ cỡ vài nghìn bước.
- **Space: `O(d²)`** cho memo + `O(d)` độ sâu đệ quy (tối đa 11 tầng, không lo giới hạn đệ quy của Python).

Cell dưới đếm số lời gọi khi **không** và **có** memo. Để thấy memo hoạt động thế nào "bên trong", ở đây ta tự viết memo bằng một dict `memo` (đúng việc `lru_cache` làm hộ ta). `nonlocal calls` cho phép hàm con sửa biến `calls` của hàm cha.

In [22]:
def digit_dp_stats(n, use_memo):
    digits = [int(c) for c in str(n)]
    L = len(digits)
    memo = {}
    calls = 0

    def dfs(pos, ones, tight):
        nonlocal calls
        calls += 1
        if pos == L:
            return ones
        key = (pos, ones, tight)
        if use_memo and key in memo:          # đã tính rồi -> lấy ra luôn
            return memo[key]
        limit = digits[pos] if tight else 9
        total = 0
        for d in range(limit + 1):
            total += dfs(pos + 1, ones + (1 if d == 1 else 0), tight and d == limit)
        memo[key] = total
        return total

    return dfs(0, 0, True), calls, len(memo)

for n in [99999, 10**9]:
    if n <= 10**5:
        ans, calls, _ = digit_dp_stats(n, use_memo=False)
        print(f"n = {n:>10}: KHÔNG memo -> đáp án {ans:>11,}, số lời gọi {calls:>9,}")
    ans, calls, states = digit_dp_stats(n, use_memo=True)
    print(f"n = {n:>10}: CÓ memo    -> đáp án {ans:>11,}, số lời gọi {calls:>9,}, số trạng thái lưu {states}")

n =      99999: KHÔNG memo -> đáp án      50,000, số lời gọi   111,111
n =      99999: CÓ memo    -> đáp án      50,000, số lời gọi       191, số trạng thái lưu 19
n = 1000000000: CÓ memo    -> đáp án 900,000,001, số lời gọi       462, số trạng thái lưu 55


In [23]:
def count_digit_one_digit_dp(n: int) -> int:
    digits = [int(c) for c in str(n)]
    L = len(digits)

    @lru_cache(maxsize=None)               # cache MỚI cho mỗi lần gọi hàm ngoài
    def dfs(pos, ones, tight):
        if pos == L:
            return ones
        limit = digits[pos] if tight else 9
        total = 0
        for d in range(limit + 1):
            total += dfs(pos + 1, ones + (1 if d == 1 else 0), tight and d == limit)
        return total

    return dfs(0, 0, True)

for n in [13, 0, 214, 1024, 10**9]:
    print(f"n = {n:>10} -> {count_digit_one_digit_dp(n)}")

n =         13 -> 6
n =          0 -> 0
n =        214 -> 147
n =       1024 -> 338
n = 1000000000 -> 900000001


## 14. Code nộp LeetCode

Bản nộp chính là **Cách 3** (3 trường hợp): nhanh nhất, ít bộ nhớ nhất, dễ giải thích nhất.

Chuyển từ hàm demo sang format LeetCode chỉ cần:

- Bọc trong `class Solution` và đặt tên method đúng `countDigitOne(self, n: int) -> int`.
- Bỏ mọi `print` (code nộp bài phải gọn).
- Không cần xử lý riêng `n = 0`: vòng `while i <= n` tự không chạy.

In [24]:
class Solution:
    def countDigitOne(self, n: int) -> int:
        ans = 0
        i = 1
        while i <= n:
            high = n // (i * 10)
            cur = (n // i) % 10
            low = n % i
            if cur == 0:
                ans += high * i
            elif cur == 1:
                ans += high * i + low + 1
            else:
                ans += (high + 1) * i
            i *= 10
        return ans

In [25]:
sol = Solution()

tests = [
    (13, 6),                     # Ví dụ 1
    (0, 0),                      # Ví dụ 2
    (1, 1),                      # nhỏ nhất có chữ số 1
    (9, 1),
    (10, 2),                     # n = 10^k: hàng cao nhất có cur = 1, low = 0
    (11, 4),                     # 11 có hai chữ số 1
    (99, 20),
    (100, 21),
    (101, 23),
    (111, 36),
    (214, 147),
    (1024, 338),                 # đủ 3 trường hợp cur = 0 / 1 / >1
    (2101, 1623),
    (999_999_999, 900_000_000),  # dấu _ chỉ để dễ đọc: 999999999
    (10**9, 900_000_001),        # biên trên của constraints
]

for n, expected in tests:
    got = sol.countDigitOne(n)
    assert got == expected, (n, got, expected)
print(f"Solution: PASS {len(tests)} test (ví dụ đề + edge case)")

Solution: PASS 15 test (ví dụ đề + edge case)


### Bản nộp thay thế — Digit DP

Cũng nộp được (chạy rất nhanh với `n ≤ 10^9`). Hai điểm cần chú ý khi đưa digit DP vào `class Solution`:

- Hàm `dfs` có `@lru_cache` phải được **định nghĩa bên trong method** `countDigitOne`, để **mỗi lần gọi** (mỗi test) có một cache mới. Nếu đặt `dfs` ở ngoài và dùng chung cache thì kết quả test sau bị "nhiễm" kết quả test trước — xem Lỗi 4 ở mục 17.
- Viết rõ `from functools import lru_cache` (môi trường LeetCode thường import sẵn, nhưng viết rõ thì chạy được ở mọi nơi).

In [26]:
from functools import lru_cache

class SolutionDigitDP:
    def countDigitOne(self, n: int) -> int:
        digits = [int(c) for c in str(n)]
        L = len(digits)

        @lru_cache(maxsize=None)
        def dfs(pos: int, ones: int, tight: bool) -> int:
            if pos == L:
                return ones
            limit = digits[pos] if tight else 9
            total = 0
            for d in range(limit + 1):
                total += dfs(pos + 1, ones + (1 if d == 1 else 0), tight and d == limit)
            return total

        return dfs(0, 0, True)

sol_dp = SolutionDigitDP()
for n, expected in tests:
    assert sol_dp.countDigitOne(n) == expected, n
print(f"SolutionDigitDP: PASS {len(tests)} test")

SolutionDigitDP: PASS 15 test


### Đối chiếu chéo các cách (test chéo)

Test bằng vài ví dụ là chưa đủ — ta cho **mọi cách đấu với trọng tài** trên thật nhiều input:

1. **Mọi `n` từ 0 đến 5000.** Gọi Cách 1 riêng cho từng `n` sẽ tốn khoảng `5000²/2` bước — quá phí. Mẹo: xây bảng **cộng dồn** `prefix[x] = prefix[x − 1] + str(x).count("1")`, vậy `prefix[n]` chính là đáp án brute force của `n`, cả bảng chỉ tốn `O(5000)`.
2. Cách 2 chậm hơn nên chỉ thử trên 100 giá trị `n` ngẫu nhiên trong `[0, 5000]`; `random.seed(233)` cố định "hạt giống" để mỗi lần chạy notebook đều ra cùng dãy số ngẫu nhiên.
3. Vài `n` cỡ `10^4 .. 2·10^5` so trực tiếp với Cách 1.
4. **`n` rất lớn** (tới `10^9`): brute force không chạy nổi, nên 3 cách nhanh phải **khớp nhau**. Thêm các giá trị "hiểm" `10^k − 1`, `10^k`, `10^k + 1`, `111111111`, `199999999`.
5. Một kiểm tra **bằng toán độc lập**: trong `[0, 10^k − 1]` (mọi chuỗi `k` chữ số, kể cả số 0 đứng đầu), mỗi hàng có chữ số 1 ở đúng `1/10` số chuỗi → tổng `k · 10^(k−1)`. Ví dụ `k = 2`: `[0, 99]` có `2 · 10 = 20` chữ số 1 (khớp test `(99, 20)` ở trên).

In [27]:
import random

# 1) Bảng đáp án brute force cho mọi n trong [0, 5000] (cộng dồn)
LIMIT = 5000
prefix = [0] * (LIMIT + 1)
for x in range(1, LIMIT + 1):
    prefix[x] = prefix[x - 1] + str(x).count("1")

fast_methods = {
    "Cách 3 (3 trường hợp)": count_digit_one_position,
    "Cách 4 (công thức gọn)": count_digit_one_formula,
    "Cách 5 (digit DP)": count_digit_one_digit_dp,
    "Solution": Solution().countDigitOne,
    "SolutionDigitDP": SolutionDigitDP().countDigitOne,
}
for name, f in fast_methods.items():
    for n in range(LIMIT + 1):
        assert f(n) == prefix[n], (name, n, f(n), prefix[n])
    print(f"{name:<23}: khớp brute force với MỌI n trong [0, {LIMIT}]")

# 2) Cách 2 trên 100 giá trị ngẫu nhiên
random.seed(233)
for _ in range(100):
    n = random.randint(0, LIMIT)
    assert count_ones_bruteforce_mod(n) == prefix[n], n
print("Cách 2 (% 10)          : khớp brute force trên 100 n ngẫu nhiên trong [0, 5000]")

# 3) Vài n lớn hơn so trực tiếp với trọng tài Cách 1
for _ in range(5):
    n = random.randint(10**4, 2 * 10**5)
    assert count_digit_one_position(n) == count_ones_bruteforce_str(n), n
print("Cách 3 khớp Cách 1 trên 5 n ngẫu nhiên trong [10^4, 2*10^5]")

# 4) n rất lớn: 3 cách nhanh phải khớp NHAU
special = [10**k + delta for k in range(1, 10) for delta in (-1, 0, 1)]
special += [111111111, 199999999]
big = [v for v in special if v <= 10**9] + [random.randint(0, 10**9) for _ in range(300)]
for n in big:
    a, b, c = count_digit_one_position(n), count_digit_one_formula(n), count_digit_one_digit_dp(n)
    assert a == b == c, (n, a, b, c)
print(f"Cách 3, 4, 5 khớp nhau trên {len(big)} giá trị lớn (kể cả 10^k - 1, 10^k, 10^k + 1)")

# 5) Kiểm tra bằng toán: f(10^k - 1) = k * 10^(k-1)
for k in range(1, 10):
    assert count_digit_one_position(10**k - 1) == k * 10**(k - 1)
print("f(10^k - 1) = k * 10^(k-1) đúng với k = 1..9")

Cách 3 (3 trường hợp)  : khớp brute force với MỌI n trong [0, 5000]
Cách 4 (công thức gọn) : khớp brute force với MỌI n trong [0, 5000]


Cách 5 (digit DP)      : khớp brute force với MỌI n trong [0, 5000]
Solution               : khớp brute force với MỌI n trong [0, 5000]


SolutionDigitDP        : khớp brute force với MỌI n trong [0, 5000]


Cách 2 (% 10)          : khớp brute force trên 100 n ngẫu nhiên trong [0, 5000]


Cách 3 khớp Cách 1 trên 5 n ngẫu nhiên trong [10^4, 2*10^5]
Cách 3, 4, 5 khớp nhau trên 328 giá trị lớn (kể cả 10^k - 1, 10^k, 10^k + 1)
f(10^k - 1) = k * 10^(k-1) đúng với k = 1..9


In [28]:
# Cảm nhận tốc độ ở biên trên n = 10^9 (trung bình qua 200 lần gọi)
for name, f in [("Cách 3", count_digit_one_position),
                ("Cách 4", count_digit_one_formula),
                ("Cách 5", count_digit_one_digit_dp)]:
    start = time.perf_counter()
    for _ in range(200):
        result = f(10**9)
    avg = (time.perf_counter() - start) / 200
    print(f"{name}: f(10^9) = {result:,} | trung bình {avg * 1e6:8.1f} micro giây / lần")

Cách 3: f(10^9) = 900,000,001 | trung bình      2.3 micro giây / lần
Cách 4: f(10^9) = 900,000,001 | trung bình      3.5 micro giây / lần
Cách 5: f(10^9) = 900,000,001 | trung bình    260.9 micro giây / lần


## 15. Vì sao Cách 3 đúng?

**Bước 1 — Tách bài lớn thành từng hàng, không trùng, không sót.** Mỗi "lần xuất hiện của chữ số 1" là một cặp *(số `x`, hàng `i`)* sao cho `0 ≤ x ≤ n` và chữ số hàng `i` của `x` bằng 1. Mỗi cặp thuộc **đúng một** hàng `i`. Vậy nhóm các cặp theo hàng thì không cặp nào bị đếm hai lần, không cặp nào bị sót:

> **Đáp án = tổng `count_i` qua mọi hàng `i`.**

**Bất biến (invariant) của vòng lặp:** *sau khi xử lý xong hàng `i`, `ans` bằng tổng số chữ số 1 nằm ở các hàng `1, 10, ..., i` của mọi số từ 0 đến `n`.* Vòng lặp dừng khi `i > n` — các hàng cao hơn đều là chữ số 0 với mọi `x ≤ n` → `ans` là đáp án đầy đủ.

**Bước 2 — Vì sao `count_i` đúng 3 công thức? (lập luận "đóng khung")** Một số `x` có chữ số 1 ở hàng `i` luôn viết được dạng:

```text
x = H · (10i) + 1 · i + L        với H ≥ 0 (khúc trái),  0 ≤ L ≤ i − 1 (khúc phải)
        [  H  ][ 1 ][  L  ]
n = high · (10i) + cur · i + low
        [high ][cur][ low ]
```

So sánh `x` với `n` giống so hai số cùng độ dài: **so khúc trái trước**.

- `H < high`: dù `L` là gì thì `x < n`. `H` có `high` lựa chọn (`0..high−1`), `L` có `i` lựa chọn → **`high · i`**.
- `H > high`: `x > n` → **0**.
- `H = high`: phải so tiếp chữ số giữa:
  - `cur > 1`: chữ số giữa của `x` là 1 < `cur` → `x < n` với mọi `L` → thêm **`i`**.
  - `cur = 1`: chữ số giữa bằng nhau → cần `L ≤ low` → `L ∈ 0..low` → thêm **`low + 1`**.
  - `cur = 0`: chữ số giữa 1 > 0 → `x > n` → thêm **0**.

Cộng lại ra đúng 3 công thức. Ví dụ `n = 214, i = 10`: `H ∈ {0, 1}` tự do → `2 · 10 = 20` số (`010..019`, `110..119`); `H = 2`, `cur = 1` → `L ∈ 0..4` → `210..214` (5 số) → `25`.

Để ý: ý "so khúc trái trước — nếu đã nhỏ hơn thì phần sau tự do" chính là **cờ `tight`** của digit DP (`H < high` ⇔ "đã thoát trần"). Hai cách nhìn khác nhau nhưng cùng một bản chất.

## 16. So sánh các cách

Ký hiệu: `n` là giá trị input, `d` là số chữ số của `n` (`d ≈ log10 n + 1`, tối đa 10).

| Cách | Ý tưởng | Time | Space | Với `n = 10^9` | Khi nào dùng |
|---|---|---|---|---|---|
| 1 | Đếm `'1'` trong `str(x)` cho mọi `x` | `O(n · d)` | `O(d)` | cỡ một phút trở lên → TLE | Trọng tài để test |
| 2 | Bóc chữ số bằng `% 10`, `// 10` | `O(n · d)` | `O(1)` | còn chậm hơn Cách 1 (vài phút) → TLE | Luyện tư duy chữ số |
| 3 | **Đếm theo hàng, 3 trường hợp** | `O(d)` | `O(1)` | 10 vòng lặp | **Cách chính**, dễ giải thích |
| 4 | Công thức gọn `min(max(...))` | `O(d)` | `O(1)` | 10 vòng lặp | Viết nhanh khi đã hiểu Cách 3 |
| 5 | Digit DP (đệ quy + memo + `tight`) | `O(10 · d²)` | `O(d²)` | vài nghìn bước | Khuôn tổng quát cho các bài biến thể |

Tóm tắt lộ trình tư duy: **duyệt từng số** (1, 2) → nhận ra phải **đếm theo cột** → mỗi cột có **chu kỳ `10i`** → **3 trường hợp** (3) → gộp lại bằng **giao hai đoạn** (4) → tổng quát hóa thành **điền ô có cờ `tight`** (5).

## 17. Các lỗi người mới hay gặp

### Lỗi 1 — Trường hợp `cur == 1` quên `+ 1`

```python
elif cur == 1:
    ans += high * i + low        # SAI
```

**Vì sao sai:** khúc phải `L` chạy từ `0` đến `low` có **`low + 1`** giá trị — phải đếm cả `L = 0`, tức chính số có dạng `high 1 000...0` (như `210`, `1000`). Với `n = 10`: hàng chục có `high = 0, cur = 1, low = 0` → phải cộng 1 (số `10`), bản sai cộng 0 → trả về 1 thay vì 2.

**Sửa:** `ans += high * i + low + 1`.

### Lỗi 2 — Điều kiện vòng lặp `while i < n`

```python
while i < n:                     # SAI, phải là i <= n
```

**Vì sao sai:** khi `n` đúng bằng `10^k` (`1, 10, 100, ...`), hàng cao nhất có `i == n` bị bỏ qua. `n = 1` → vòng lặp không chạy → trả về 0 (đúng là 1).

Lỗi này **rất khó phát hiện** nếu chỉ test bằng ví dụ của đề: với `n = 13` (và cả `n = 214`) bản sai vẫn ra đúng! Nó chỉ lộ ra khi test đúng các lũy thừa của 10 — đó là lý do bộ test ở mục 14 có `1, 10, 100`.

In [29]:
def wrong_forget_plus_one(n):
    ans, i = 0, 1
    while i <= n:
        high, cur, low = n // (i * 10), (n // i) % 10, n % i
        if cur == 0:
            ans += high * i
        elif cur == 1:
            ans += high * i + low              # SAI: thiếu + 1
        else:
            ans += (high + 1) * i
        i *= 10
    return ans

def wrong_loop_condition(n):
    ans, i = 0, 1
    while i < n:                               # SAI: phải là i <= n
        high, cur, low = n // (i * 10), (n // i) % 10, n % i
        if cur == 0:
            ans += high * i
        elif cur == 1:
            ans += high * i + low + 1
        else:
            ans += (high + 1) * i
        i *= 10
    return ans

for n in [1, 10, 13, 100, 214]:
    print(f"n = {n:>3}: đúng = {count_digit_one_position(n):>3} | quên +1 = {wrong_forget_plus_one(n):>3}"
          f" | while i < n = {wrong_loop_condition(n):>3}")

n =   1: đúng =   1 | quên +1 =   0 | while i < n =   0
n =  10: đúng =   2 | quên +1 =   1 | while i < n =   1
n =  13: đúng =   6 | quên +1 =   5 | while i < n =   6
n = 100: đúng =  21 | quên +1 =  20 | while i < n =  20
n = 214: đúng = 147 | quên +1 = 146 | while i < n = 147


### Lỗi 3 — Dùng `/` thay cho `//`

Người quen toán phổ thông hay viết `n / 10` để "chia cho 10". Trong Python 3, `/` là **chia thực**, luôn ra `float`:

- `214 / 10 = 21.4`, nên `cur = (214 / 10) % 10` ra khoảng `1.4` (máy in ra `1.3999999999999986` vì sai số làm tròn của `float`) — **không** bằng 1! Code rơi nhầm vào nhánh `cur > 1`.
- `high = 214 / 100 = 2.14` cũng không còn là số nguyên → kết quả ra số lẻ thập phân vô nghĩa.

Ngoài ra `float` chỉ chính xác tới khoảng `2^53`, nên với số rất lớn còn sai cả khi đã ép `int(...)`. **Quy tắc:** xử lý chữ số thì **luôn** dùng `//` và `%`.

In [30]:
n, i = 214, 10
print("Dùng / :", "high =", n / (i * 10), "| cur =", (n / i) % 10)
print("Dùng //:", "high =", n // (i * 10), "| cur =", (n // i) % 10)

def wrong_true_division(n):
    ans, i = 0, 1
    while i <= n:
        high, cur, low = n / (i * 10), (n / i) % 10, n % i      # SAI: dùng /
        if cur == 0:
            ans += high * i
        elif cur == 1:
            ans += high * i + low + 1
        else:
            ans += (high + 1) * i
        i *= 10
    return ans

print("Kết quả sai (dùng /) :", wrong_true_division(214))
print("Kết quả đúng (dùng //):", count_digit_one_position(214))

Dùng / : high = 2.14 | cur = 1.3999999999999986
Dùng //: high = 2 | cur = 1
Kết quả sai (dùng /) : 175.2
Kết quả đúng (dùng //): 147


### Lỗi 4 — Digit DP: dùng chung memo giữa các lần gọi với `n` khác nhau

Một cách viết "tiện tay": đặt `dfs` có `@lru_cache` ở **ngoài cùng**, đọc chữ số của `n` từ biến toàn cục `DIGITS`.

**Vì sao sai:** key của cache chỉ là `(pos, ones, tight)` — **không chứa thông tin gì về `n`**. Gọi cho `n = 13` xong, cache đã có `dfs(0, 0, True) = 6`. Gọi tiếp cho `n = 25` (cũng 2 chữ số), `dfs(0, 0, True)` trúng cache → trả luôn `6` cũ, trong khi đáp án đúng là 13.

Trên LeetCode lỗi này rất "ác": nhiều test chạy chung một tiến trình, test đầu đúng, các test sau sai.

**Sửa:** định nghĩa `dfs` **bên trong** hàm chính như Cách 5 (mỗi lần gọi có cache mới); hoặc gọi `dfs.cache_clear()` trước mỗi `n`; hoặc đưa thông tin của `n` vào key.

In [31]:
DIGITS = []

@lru_cache(maxsize=None)
def dfs_global(pos, ones, tight):
    if pos == len(DIGITS):
        return ones
    limit = DIGITS[pos] if tight else 9
    total = 0
    for d in range(limit + 1):
        total += dfs_global(pos + 1, ones + (1 if d == 1 else 0), tight and d == limit)
    return total

def wrong_digit_dp_shared_cache(n):
    global DIGITS
    DIGITS = [int(c) for c in str(n)]
    return dfs_global(0, 0, True)          # SAI: cache còn giữ kết quả của n trước

for n in [13, 25, 99]:
    print(f"n = {n}: memo dùng chung -> {wrong_digit_dp_shared_cache(n):>2} | đúng -> {count_digit_one_digit_dp(n)}")

n = 13: memo dùng chung ->  6 | đúng -> 6
n = 25: memo dùng chung ->  6 | đúng -> 13
n = 99: memo dùng chung ->  6 | đúng -> 20


### Các lỗi khác cần nhớ

- **Đếm "số lượng số chứa 1"** thay vì "số lần chữ số 1 xuất hiện" (số `11` phải tính 2 lần) — xem lại mục 1.
- Brute force viết `range(n)` thay vì `range(n + 1)` → **bỏ sót chính `n`** (`n = 1` ra 0).
- Digit DP **quên cờ `tight`** (cho `d` luôn chạy `0..9`) → đếm mọi số có `d` chữ số, vượt quá `n`: với `n = 13` sẽ đếm cả `00..99` và ra 20.
- Digit DP viết `tight_mới = (d == limit)` (thiếu `tight and`) → đã thoát trần rồi mà gặp `d == 9` lại bị coi là "dính trần" → sai.
- Nghĩ phải xử lý riêng `n = 0` bằng `if` — không cần, nhưng nếu thêm thì cũng không sai.

## 18. Mẹo nhận dạng pattern

- **Dấu hiệu:** đề hỏi "đếm / tính tổng một tính chất về **chữ số**" trên **mọi số từ 0 (hoặc 1) đến `n`**, và `n` rất lớn (`10^9`, `10^15`, `10^18`) → **không được liệt kê** từng số.
  - Tính chất là "chữ số `d` xuất hiện bao nhiêu lần" → **đếm theo từng hàng** (high / cur / low).
  - Tính chất phức tạp hơn (chữ số không lặp, tổng chữ số, không có hai số 1 liền nhau, chia hết cho `k`, ...) → **digit DP** với `dfs(pos, trạng_thái, tight)`.
- Đề hỏi trên đoạn `[L, R]` → tính **`f(R) − f(L − 1)`** với `f(x)` là đáp án trên `[0, x]` (mẹo "prefix").
- Ba hình ảnh nên nhớ: **chu kỳ `10·i`** của mỗi hàng; **"đóng khung" `[H][1][L]`**; **cờ `tight`** = "còn dính trần hay đã thoát trần".

## 19. Biến thể và bài liên quan

| Bài | Độ khó | Liên hệ với bài này |
|---|---|---|
| Đếm chữ số `d` bất kỳ (`1 ≤ d ≤ 9`) trong `[0, n]` | — | Thay `1` bằng `d`: so `cur` với `d` (`cur < d` / `cur == d` / `cur > d`) — Bài tập 4 |
| Đếm chữ số **0** trong `[1, n]` | — | Khó hơn: số 0 **đứng đầu** (leading zero) không được tính (`7` không có chữ số 0 nào). Cách hàng: phần `high` phải bắt đầu từ 1; digit DP: thêm cờ `started` |
| LeetCode 1067 — Digit Count in Range | Hard (Premium) | Đếm chữ số `d` trong `[low, high]` = `f(high) − f(low − 1)` |
| LeetCode 902 — Numbers At Most N Given Digit Set | Hard | Digit DP với `tight` + xử lý số ngắn hơn `n` |
| LeetCode 600 — Non-negative Integers without Consecutive Ones | Hard | Digit DP trên hệ **nhị phân**, trạng thái = bit trước đó |
| LeetCode 1012 — Numbers With Repeated Digits | Hard | Digit DP + bitmask các chữ số đã dùng (đếm phần bù) |
| LeetCode 2376 — Count Special Integers | Hard | Đếm số có các chữ số đôi một khác nhau |
| LeetCode 2719 — Count of Integers | Hard | Digit DP + trạng thái tổng chữ số, trên đoạn `[num1, num2]` |
| LeetCode 357 — Count Numbers with Unique Digits | Medium | Bài khởi động tổ hợp cho 2376 |
| LeetCode 400 — Nth Digit | Medium | Cũng "đếm theo khối chữ số" |
| Cracking the Coding Interview 17.6 — Count of 2s | — | Y hệt bài này nhưng với chữ số 2 |

## 20. Bài tập vận dụng

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 21) và **Đáp án** (mục 22, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

```python
n, i = 3141, 100
print(n // (i * 10), (n // i) % 10, n % i)          # (a)
print(sum(str(x).count("1") for x in range(12)))     # (b)
print("1111".count("11"))                            # (c)
print(min(max(57 - 100 + 1, 0), 100))                # (d)
```

(e) Với `n = 3141`, hàng trăm góp bao nhiêu chữ số 1 vào đáp án?

### Bài 2 — Tính tay bằng bảng high / cur / low

(a) Lập bảng như mục 11 (cột `i`, `high`, `cur`, `low`, trường hợp, cộng thêm, `ans`) cho **`n = 2101`** và tính đáp án.

(b) Làm tương tự cho **`n = 99`**.

Chỉ dùng giấy bút; phần đáp án có code kiểm tra.

### Bài 3 — Điền TODO cho cách chính

Hoàn thiện `exercise_count_digit_one(n)` theo Cách 3, không nhìn lại mục 10.

In [32]:
def exercise_count_digit_one(n: int) -> int:
    ans = 0
    i = 1
    # TODO 1: while ...:            (điều kiện: hàng i còn tồn tại trong n — nhớ Lỗi 2)
    #     TODO 2: high = ..., cur = ..., low = ...
    #     TODO 3: cộng vào ans theo 3 trường hợp cur == 0 / cur == 1 / cur > 1
    #     TODO 4: i = ...           (sang hàng kế tiếp)
    raise NotImplementedError("Hãy tự hoàn thành Bài 3")

# Tự làm xong rồi mới mở test:
# assert exercise_count_digit_one(13) == 6
# assert exercise_count_digit_one(0) == 0
# assert exercise_count_digit_one(10) == 2
# assert exercise_count_digit_one(1024) == 338
# assert exercise_count_digit_one(10**9) == 900_000_001

### Bài 4 — Biến thể: đếm chữ số `d` bất kỳ (`1 ≤ d ≤ 9`)

Viết `exercise_count_digit_d(n, d)` trả về tổng số lần chữ số `d` xuất hiện trong các số `0, 1, ..., n`.

```python
count_digit_d(13, 1) -> 6      # chính là bài gốc
count_digit_d(25, 2) -> 9      # 2, 12, 20, 21, 22 (hai lần), 23, 24, 25
count_digit_d(100, 9) -> 20
```

(Chưa xét `d = 0` vì số 0 đứng đầu làm bài khác hẳn — xem mục 19.)

In [33]:
def exercise_count_digit_d(n: int, d: int) -> int:
    # TODO: giữ nguyên khung Cách 3, chỉ đổi cách so sánh cur
    pass

# Gợi ý test:
# assert exercise_count_digit_d(13, 1) == 6
# assert exercise_count_digit_d(25, 2) == 9
# assert exercise_count_digit_d(100, 9) == 20

### Bài 5 — Biến thể: đếm chữ số 1 trong đoạn `[low, high]`

Viết `exercise_count_ones_in_range(low, high)` với `0 ≤ low ≤ high ≤ 10^9`: tổng số lần chữ số 1 xuất hiện trong các số `low, low + 1, ..., high`.

```python
count_ones_in_range(10, 13) -> 5     # 10, 11 (hai lần), 12, 13
count_ones_in_range(11, 11) -> 2
count_ones_in_range(0, 13)  -> 6
count_ones_in_range(0, 0)   -> 0
```

Yêu cầu: chạy nhanh cả với `high = 10^9`.

In [34]:
def exercise_count_ones_in_range(low: int, high: int) -> int:
    # TODO: dùng lại một hàm đếm trên [0, x] đã viết
    pass

# Gợi ý test:
# assert exercise_count_ones_in_range(10, 13) == 5
# assert exercise_count_ones_in_range(11, 11) == 2
# assert exercise_count_ones_in_range(0, 13) == 6
# assert exercise_count_ones_in_range(0, 0) == 0

### Bài 6 — Mini challenge (digit DP): có bao nhiêu số trong `[1, n]` chứa **ít nhất một** chữ số 1?

Đây chính là "cách hiểu sai đề" ở mục 1 — giờ trở thành một bài thật! Với `n` tới `10^9` thì **không** được duyệt từng số.

```python
count_numbers_with_one(13)  -> 5     # 1, 10, 11, 12, 13
count_numbers_with_one(100) -> 20    # 1, 10..19, 21, 31, ..., 91, 100
count_numbers_with_one(0)   -> 0
```

Gợi ý nhỏ: tính chất "có ít nhất một chữ số 1" không đếm được theo từng hàng một cách độc lập (số `11` chỉ được tính một lần) → đây đúng là lúc dùng khuôn digit DP.

In [35]:
def exercise_count_numbers_with_one(n: int) -> int:
    # TODO: digit DP với trạng thái (pos, has_one, tight)
    pass

# Gợi ý test:
# assert exercise_count_numbers_with_one(13) == 5
# assert exercise_count_numbers_with_one(100) == 20
# assert exercise_count_numbers_with_one(0) == 0

## 21. Gợi ý theo mức

**Bài 1**
1. Nhớ lại bảng định nghĩa `high`, `cur`, `low` ở mục 4.
2. (c): `.count` đếm các lần xuất hiện **không chồng lấn** nhau, quét từ trái sang phải.
3. (d) chính là phần "kẹp" của Cách 4 với `r = 57`, `i = 100`; (e) dùng kết quả (a) và trường hợp `cur = 1`.

**Bài 2**
1. Lần lượt `i = 1, 10, 100, 1000` (dừng khi `i > n`).
2. Mỗi dòng: tính 3 khúc → nhìn `cur` để chọn trường hợp → cộng dồn.
3. Với `2101`: hàng chục có `cur = 0` nên chỉ còn `high · i = 21 · 10`; hàng đơn vị và hàng trăm có `cur = 1`.

**Bài 3**
1. Hàng `i` tồn tại khi `i <= n`.
2. `high = n // (i * 10)`, `cur = (n // i) % 10`, `low = n % i`.
3. Ba nhánh `if cur == 0 / elif cur == 1 / else`, cuối vòng `i *= 10`.

**Bài 4**
1. Chỉ phần "chu kỳ dở dang" thay đổi: so `cur` với `d` thay vì với `1`.
2. `cur < d` → chưa chạm khối (thêm 0); `cur == d` → giữa khối (thêm `low + 1`); `cur > d` → trọn khối (thêm `i`).
3. Phần `high * i` vẫn luôn có; vì `d ≥ 1` nên điều kiện `while i <= n` vẫn đúng.

**Bài 5**
1. "Đếm trong `[low, high]`" = "đếm trong `[0, high]`" trừ "đếm trong `[0, low − 1]`".
2. Cẩn thận `low = 0`: khi đó không có gì để trừ.
3. `return f(high) - (f(low - 1) if low > 0 else 0)` với `f` là một trong các hàm nhanh đã viết.

**Bài 6**
1. Giữ nguyên khuôn `dfs(pos, ..., tight)`; thay biến đếm `ones` bằng cờ `has_one` (đã gặp chữ số 1 chưa).
2. Base case: trả `1` nếu `has_one` là `True`, ngược lại `0` (đếm **số**, không đếm chữ số).
3. Chuyển trạng thái: `has_one or d == 1`. Cách khác: đếm số **không** có chữ số 1 trong `[0, n]` rồi lấy `(n + 1)` trừ đi.

## 22. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

- (a) `3 1 41` — `high = 3141 // 1000 = 3`, `cur = 31 % 10 = 1`, `low = 3141 % 100 = 41`.
- (b) `4` — các số `0..11`: `1` (1 lần), `10` (1 lần), `11` (2 lần).
- (c) `2` — `"1111"` tách được hai đoạn `"11"` **không chồng lấn**: `"11" + "11"` (không phải 3).
- (d) `0` — `57 − 100 + 1 = −42`, kẹp về `0`: `r = 57 < i = 100`, chu kỳ dở dang chưa chạm khối.
- (e) `cur = 1` → `high · i + low + 1 = 3 · 100 + 41 + 1 = 342`.

### Đáp án Bài 2

(a) `n = 2101`:

| Bước | `i` | `high` | `cur` | `low` | Trường hợp | Cộng thêm | `ans` |
|---:|---:|---:|---:|---:|---|---|---:|
| 1 | 1 | 210 | 1 | 0 | `cur = 1` | `210 · 1 + 0 + 1 = 211` | 211 |
| 2 | 10 | 21 | 0 | 1 | `cur = 0` | `21 · 10 = 210` | 421 |
| 3 | 100 | 2 | 1 | 1 | `cur = 1` | `2 · 100 + 1 + 1 = 202` | 623 |
| 4 | 1000 | 0 | 2 | 101 | `cur ≥ 2` | `(0 + 1) · 1000 = 1000` | **1623** |

(b) `n = 99`: `i = 1`: `high = 9, cur = 9` → `(9 + 1) · 1 = 10`; `i = 10`: `high = 0, cur = 9` → `(0 + 1) · 10 = 10`. Tổng **20** (khớp công thức `k · 10^(k−1)` với `k = 2`).

### Đáp án Bài 3 → Bài 6 (code bên dưới, có assert)

In [36]:
# ---------- Kiểm tra lại Bài 1 và Bài 2 bằng code ----------
n, i = 3141, 100
assert (n // (i * 10), (n // i) % 10, n % i) == (3, 1, 41)
assert sum(str(x).count("1") for x in range(12)) == 4
assert "1111".count("11") == 2
assert min(max(57 - 100 + 1, 0), 100) == 0
assert count_at_position_formula(3141, 100) == 342 == count_one_at_position_brute(3141, 100)
assert count_digit_one_position(2101) == 1623 == count_ones_bruteforce_str(2101)
assert count_digit_one_position(99) == 20

# ---------- Bài 3 ----------
def answer_count_digit_one(n: int) -> int:
    ans = 0
    i = 1
    while i <= n:
        high, cur, low = n // (i * 10), (n // i) % 10, n % i
        if cur == 0:
            ans += high * i
        elif cur == 1:
            ans += high * i + low + 1
        else:
            ans += (high + 1) * i
        i *= 10
    return ans

assert answer_count_digit_one(13) == 6
assert answer_count_digit_one(0) == 0
assert answer_count_digit_one(10) == 2
assert answer_count_digit_one(1024) == 338
assert answer_count_digit_one(10**9) == 900_000_001

# ---------- Bài 4 ----------
def answer_count_digit_d(n: int, d: int) -> int:
    ans = 0
    i = 1
    while i <= n:
        high, cur, low = n // (i * 10), (n // i) % 10, n % i
        if cur < d:
            ans += high * i
        elif cur == d:
            ans += high * i + low + 1
        else:
            ans += (high + 1) * i
        i *= 10
    return ans

assert answer_count_digit_d(13, 1) == 6
assert answer_count_digit_d(25, 2) == 9
assert answer_count_digit_d(100, 9) == 20
for d in range(1, 10):                          # đối chiếu brute force cộng dồn
    acc = 0
    for x in range(2001):
        acc += str(x).count(str(d))
        assert answer_count_digit_d(x, d) == acc, (x, d)

# ---------- Bài 5 ----------
def answer_count_ones_in_range(low: int, high: int) -> int:
    return count_digit_one_position(high) - (count_digit_one_position(low - 1) if low > 0 else 0)

assert answer_count_ones_in_range(10, 13) == 5
assert answer_count_ones_in_range(11, 11) == 2
assert answer_count_ones_in_range(0, 13) == 6
assert answer_count_ones_in_range(0, 0) == 0
random.seed(5)
for _ in range(200):
    a = random.randint(0, 3000)
    b = random.randint(a, 3000)
    assert answer_count_ones_in_range(a, b) == sum(str(x).count("1") for x in range(a, b + 1))

# ---------- Bài 6 ----------
def answer_count_numbers_with_one(n: int) -> int:
    digits = [int(c) for c in str(n)]
    L = len(digits)

    @lru_cache(maxsize=None)
    def dfs(pos, has_one, tight):
        if pos == L:
            return 1 if has_one else 0          # đếm SỐ, không đếm chữ số
        limit = digits[pos] if tight else 9
        total = 0
        for d in range(limit + 1):
            total += dfs(pos + 1, has_one or d == 1, tight and d == limit)
        return total

    return dfs(0, False, True)

assert answer_count_numbers_with_one(13) == 5
assert answer_count_numbers_with_one(100) == 20
assert answer_count_numbers_with_one(0) == 0
acc = 0
for x in range(1, 3001):                        # đối chiếu brute force cộng dồn
    acc += ("1" in str(x))
    assert answer_count_numbers_with_one(x) == acc, x
# Kiểm tra bằng toán: [0, 10^9 - 1] có 9^9 số KHÔNG chứa chữ số 1; cộng thêm chính số 10^9
assert answer_count_numbers_with_one(10**9) == (10**9 - 9**9) + 1

print("Tất cả đáp án Bài 1 -> Bài 6: PASS")
print("Ví dụ Bài 6: số lượng số trong [1, 10^9] chứa chữ số 1 =", f"{answer_count_numbers_with_one(10**9):,}")

Tất cả đáp án Bài 1 -> Bài 6: PASS
Ví dụ Bài 6: số lượng số trong [1, 10^9] chứa chữ số 1 = 612,579,512


## 23. Template ngắn dễ nhớ

**Cách chính — đếm theo hàng:**

```python
class Solution:
    def countDigitOne(self, n: int) -> int:
        ans, i = 0, 1
        while i <= n:                                              # (1) hàng i còn tồn tại
            high, cur, low = n // (i * 10), (n // i) % 10, n % i   # (2) cắt 3 khúc
            if cur == 0:
                ans += high * i                                    # (3a) chưa chạm khối
            elif cur == 1:
                ans += high * i + low + 1                          # (3b) giữa khối: +low+1
            else:
                ans += (high + 1) * i                              # (3c) trọn khối
            i *= 10                                                # (4) sang hàng kế
        return ans
```

Câu thần chú: **"Cắt ba khúc high | cur | low — chu kỳ đầy cho `high·i` — khúc dở dang thêm 0, `low + 1` hoặc `i`."**

**Khuôn digit DP tổng quát** (thay `state` và base case theo từng bài):

```python
digits = [int(c) for c in str(n)]
@lru_cache(maxsize=None)                     # đặt BÊN TRONG hàm chính
def dfs(pos, state, tight):
    if pos == len(digits):
        return <giá trị của một số hoàn chỉnh theo state>
    limit = digits[pos] if tight else 9
    return sum(dfs(pos + 1, <state mới sau khi điền d>, tight and d == limit)
               for d in range(limit + 1))
answer = dfs(0, <state ban đầu>, True)
```

## 24. Checklist trước khi submit

- [ ] Đếm **số lần** chữ số 1 (số `11` tính 2 lần), không phải số lượng số chứa 1?
- [ ] Vòng lặp là `while i <= n` (có dấu `=`), đã thử `n = 1, 10, 100`?
- [ ] Nhánh `cur == 1` có đủ `low + 1`?
- [ ] Mọi phép chia đều là `//` (không có `/`)?
- [ ] `n = 0` trả về `0`? `n = 10^9` trả về `900000001`?
- [ ] Nếu dùng digit DP: `dfs` + `lru_cache` nằm **bên trong** method, cờ cập nhật là `tight and d == limit`?
- [ ] Đúng signature `countDigitOne(self, n: int) -> int`, không còn `print` debug?